In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors as mcolors
import numpy as np
import math
import random
import sympy
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [2]:
def generator_gap(x):
    return (-1/x)*math.log(np.random.random())#random.uniform(0,1)

In [3]:
class RequestCar:
    def __init__(self, id, input_time):
        self.id = id  # Уникальный идентификатор заявки
        self.input_time = input_time  # Время поступления заявки
        self.start_service_time = None  # Время начала обслуживания
        self.served_in = []  # Каналы обслуживания, через которые прошла заявка
        self.rejected = False  # Флаг отказа в обслуживании
        self.output_time = None  # Время завершения обслуживания
        self.queue_time = 0  # Общее время ожидания в очереди
        self.service_time = 0  # Общее время обслуживания
        self.total_time_in_system = 0  # Время пребывания в системе

    def set_service_start(self, start_time, channel):
        """Фиксирует начало обслуживания заявки."""
        self.start_service_time = start_time
        self.served_in.append(channel)
        self.queue_time = start_time - self.input_time  # Время в очереди

    def set_service_end(self, end_time):
        """Фиксирует завершение обслуживания заявки."""
        self.output_time = end_time
        self.service_time = end_time - self.start_service_time
        self.total_time_in_system = end_time - self.input_time

    def mark_rejected(self):
        """Помечает заявку как отклоненную."""
        self.rejected = True
        self.total_time_in_system = 0  # Заявка не обслужена, значит, не находилась в системе
    def __str__(self):
        """Вывод информации о заявке."""
        status = "Отклонена" if self.rejected else "Обслужена"
        return (f"Заявка {self.id}\t: Время поступления {self.input_time}, "
                f"Статус: {status}, Время ожидания: {self.queue_time}, "
                f"Обслуживалась в: {self.served_in}, Время завершения: {self.output_time}")

In [4]:
def service_class(requests, number_queue,kanal_mu):
    kanal_mu = kanal_mu  # Интенсивность обслуживанmия
    request_objects = [RequestCar(idx,requests[idx]) for idx in range(len(requests))]

    table = {
        "запросы": requests,
        "канал 1": [],
        "канал 2": [],
        "обслуженно": [],
        "отказ": [],
    }

    for i in range(number_queue):
        table[f"очередь {i+1}"] = []

    choice_col = list(table.keys())[2:0:-1]
    choice_queue = list(table.keys())[5:]

    for req_obj in request_objects:
        req = req_obj.input_time
        is_served = False

        for kanal in choice_col:
            if not table[kanal] or table[kanal][-1][1] <= req:
                start_time = req
                end_time = round(req + generator_gap(kanal_mu[kanal]), 3)
                table[kanal].append((start_time, end_time))
                table['обслуженно'].append((table[kanal][-1], req_obj.id))

                req_obj.set_service_start(start_time, kanal)
                req_obj.set_service_end(end_time)
                is_served = True
                break

        if is_served:
            continue

        near_time = min(table["канал 1"][-1][1], table["канал 2"][-1][1])
        near_kanal = "канал 2" if table["канал 2"][-1][1] == near_time else "канал 1"

        if number_queue==0 or table[f"очередь {number_queue}"]:
            if number_queue==0 or table[f"очередь {number_queue}"][-1][1] >= req:
                table['отказ'].append((req, req_obj.id))
                req_obj.mark_rejected()
                continue

        near_queue = number_queue
        for i in range(number_queue - 1, 0, -1):
            if not table[f"очередь {i}"] or table[f"очередь {i}"][-1][1] <= req:
                near_queue = i

        enum = [near_kanal] + choice_queue[:near_queue]
        enum.reverse()
        prev = req

        for i in range(1, len(enum)):
            if not table[enum[i - 1]] or table[enum[i - 1]][-1][1]:
                a = (prev, table[enum[i]][-1][1])
                table[enum[i - 1]].append(a)
                prev = table[enum[i]][-1][1]

        start_time = prev
        end_time = round(near_time + generator_gap(kanal_mu[near_kanal]), 3)
        table[near_kanal].append((start_time, end_time))
        table['обслуженно'].append((table[near_kanal][-1], req_obj.id))

        req_obj.set_service_start(start_time, near_kanal)
        req_obj.set_service_end(end_time)

    return table, request_objects
def filter_table_by_time_range(table, start_time, end_time):
    """
    Фильтрует данные в таблице, оставляя только те, которые попадают в указанный временной диапазон.

    Параметры:
        table (dict): Исходная таблица с данными
        start_time (float): Начальное время диапазона
        end_time (float): Конечное время диапазона

    Возвращает:
        dict: Отфильтрованная таблица
    """
    filtered_table = {}

    for key, data_list in table.items():
        filtered_data = []

        for item in data_list:
            # Определяем временные границы элемента
            if isinstance(item, (int, float)):
                # Для простых временных меток
                time = item
                time_start = time_end = time
            elif len(item) == 2:
                if isinstance(item[0], (int, float)):
                    # Для пар (время, ID)
                    time = item[0]
                    time_start = time_end = time
                else:
                    # Для интервалов (start, end)
                    time_start, time_end = item[0]
            else:
                continue  # Пропускаем неподдерживаемые форматы

            # Проверяем попадание в диапазон
            if  end_time >= time_end >= start_time and end_time >= time_start >= start_time:
                filtered_data.append(item)

        filtered_table[key] = filtered_data

    return filtered_table
def visualize_service_system(table, num_queues=3):
    fig, ax = plt.subplots(figsize=(100, 6))

    # Настройки внешнего вида
    colors = {
        'запросы': 'lightgray',
        'канал 1': 'lightblue',
        'канал 2': 'lightgreen',
        'очередь 1': 'mistyrose',
        'очередь 2': 'peachpuff',
        'очередь 3': 'lavender',
        'отказ': 'red',
        'обслуженно': 'limegreen'  # Новый цвет для обслуженных заявок
    }

    # Определение вертикальных позиций для каждой строки
    y_positions = {
        'запросы': 7,  # Сдвигаем вверх на 1
        'обслуженно': 6,  # Новая строка для обслуженных заявок
        'канал 1': 5,
        'канал 2': 4,
        'очередь 1': 3,
        'очередь 2': 2,
        'очередь 3': 1,
        'отказ': 0
    }

    # Отрисовка заявок (временные точки с индексами)
    for idx, time in enumerate(table['запросы']):
        # Точка заявки
        ax.plot(time, y_positions['запросы'], 'o', color=colors['запросы'], markersize=6)

        # Текст с индексом и временем
        ax.text(time, y_positions['запросы'] + 0.2, f"{idx}\n{time:.2f}",
                ha='center', va='bottom', fontsize=7)

        # Вертикальная пунктирная линия через всю высоту графика
        ax.axvline(x=time, color='gray', linestyle=':', alpha=0.6, linewidth=0.8)

    # Отрисовка обслуженных заявок
    for segment, req_id in table['обслуженно']:
        start, end = segment
        # Прямоугольник для периода обслуживания
        ax.add_patch(patches.Rectangle(
            (start, y_positions['обслуженно'] - 0.4), end-start, 0.8,
            facecolor=colors['обслуженно'], edgecolor='black', alpha=0.6
        ))
        # Текст с ID заявки в центре отрезка
        ax.text((start + end)/2, y_positions['обслуженно'],
                f"{req_id}", ha='center', va='center', fontsize=8)

    # Функция для отображения временных меток на отрезках
    def draw_segment_labels(start, end, y_pos, color='black'):
        duration = end - start
        mid_x = (start + end) / 2

        # Время начала
        ax.text(start, y_pos - 0.3, f"{start:.2f}",
                ha='left', va='top', fontsize=7, color=color)

        # Время конца
        ax.text(end, y_pos - 0.3, f"{end:.2f}",
                ha='right', va='top', fontsize=7, color=color)

        # Длительность в центре
        ax.text(mid_x, y_pos, f"{duration:.2f}",
                ha='center', va='center', fontsize=8, color=color, weight='bold')

    # Отрисовка каналов обслуживания с подписями
    for channel in ['канал 1', 'канал 2']:
        for start, end in table[channel]:
            ax.add_patch(patches.Rectangle(
                (start, y_positions[channel] - 0.4), end-start, 0.8,
                facecolor=colors[channel], edgecolor='black'
            ))
            draw_segment_labels(start, end, y_positions[channel])

    # Отрисовка очередей с подписями
    for queue in [f'очередь {i+1}' for i in range(num_queues)]:
        if queue in table:
            for start, end in table[queue]:
                ax.add_patch(patches.Rectangle(
                    (start, y_positions[queue] - 0.4), end-start, 0.8,
                    facecolor=colors[queue], edgecolor='black'
                ))
                draw_segment_labels(start, end, y_positions[queue])

    # Отрисовка отказов
    for time, req_id in table['отказ']:
        ax.plot(time, y_positions['отказ'], 'ro', markersize=6)
        ax.text(time, y_positions['отказ'] - 0.3, f"{req_id}\n{time:.2f}",
                ha='center', va='top', fontsize=7, color='red')

    # Настройка осей и подписей
    ax.set_yticks([y_positions[k] for k in y_positions])
    ax.set_yticklabels(['Заявки', 'Обслуженно', '1 канал', '2 канал',
                       '1 место', '2 место', '3 место', 'Отказ'])
    ax.set_xlabel('Время (Тн)')
    ax.set_title('Схема обслуживания заявок с временными метками')

    # Настройка сетки
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.grid(which='major', linestyle='-', alpha=0.8)
    plt.minorticks_on()
    plt.tight_layout()

    plt.show()


In [5]:

def print_table(table, request_objects=None):
    for i in zip(table.keys(),table.values()):
        print(len(i[1]),i)
    if not request_objects:
        return
    for req_obj in request_objects:
        print(req_obj)


Отфильтрованно

In [6]:
def merge_intervals(intervals):
    """Объединяет пересекающиеся интервалы в один список."""
    if not intervals:
        return []
    intervals.sort()
    merged = [tuple(intervals[0])]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:  # Перекрытие
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged

def get_overlap(intervals1, intervals2):
    """Находит пересечение интервалов двух списков."""
    overlap = []
    i, j = 0, 0
    while i < len(intervals1) and j < len(intervals2):
        start1, end1 = intervals1[i]
        start2, end2 = intervals2[j]
        if end1 <= start2:
            i += 1
        elif end2 <= start1:
            j += 1
        else:  # Есть пересечение
            overlap.append((max(start1, start2), min(end1, end2)))
            if end1 < end2:
                i += 1
            else:
                j += 1
    return overlap
def get_free_intervals(busy_intervals, total_time):
    """Вычисляет интервалы простоя по интервалам занятости."""
    if not busy_intervals:
        return [(0, total_time)]

    free_intervals = []
    prev_end = 0

    for start, end in sorted(busy_intervals):
        if start > prev_end:
            free_intervals.append((prev_end, start))
        prev_end = max(prev_end, end)

    if prev_end < total_time:
        free_intervals.append((prev_end, total_time))

    return free_intervals

In [7]:
def probability_service1(table):
    probability=round(len(table['обслуженно'])/len(table['запросы']),3)
    print(f"Вероятность обслуживания: {probability*100} %")
    return probability
def system_throughput2(table,time_N):
    throughput=round(len(table['обслуженно'])/time_N,3)
    print(f"Пропускная способность системы: {throughput} [шт/час]")
    return throughput
def probability_failure3(table):
    probability=round(len(table['отказ'])/len(table['запросы']),3)
    print(f"Вероятность отказа: {probability*100} %")
    return probability
def single_channel_occupancy45(table, time_N):
    """
    Вычисляет вероятность занятости только одного канала в заданный период.

    :param channel1_intervals: список кортежей (start, end) для 1-го канала
    :param channel2_intervals: список кортежей (start, end) для 2-го канала
    :param time_N: общее время наблюдения
    :return: вероятность занятости только одного канала
    """
    # Объединяем интервалы занятости
    merged_channel1 = merge_intervals(table['канал 1'])
    merged_channel2 = merge_intervals(table['канал 2'])

    # Вычисляем общую занятость каждого канала
    total_channel1 = sum(end - start for start, end in merged_channel1)
    total_channel2 = sum(end - start for start, end in merged_channel2)

    # Вычисляем пересечение (время, когда оба канала заняты одновременно)
    overlap_intervals = get_overlap(merged_channel1, merged_channel2)
    total_overlap = sum(end - start for start, end in overlap_intervals)

    single_channel_time = (total_channel1 + total_channel2 - 2 * total_overlap) / time_N
    print(f"Вероятность занятости одного канала: {single_channel_time*100} %")

    second_channel_time=total_overlap / time_N
    print(f"Вероятность занятости двух канала: {second_channel_time*100} %")

    return single_channel_time,second_channel_time
def average_occupied_channels6(single_channel_occupancy45):
    buf=1*single_channel_occupancy45[0]+2*single_channel_occupancy45[1]
    print(f"Среднее количество занятых каналов: {buf} канала\tПоказатель загрузки: {buf/2}%")
    return buf
def calculate_idle_probabilities789(table, time_N):
    """
    Вычисляет вероятность простоя хотя бы одного канала, двух каналов одновременно и всей системы.

    :param table: словарь с данными системы (должен содержать 'канал 1' и 'канал 2')
    :param time_N: общее время наблюдения
    :return: (P1*, P2*, Pc*) - вероятности простоя
    """
    # Объединяем и сортируем интервалы занятости для каждого канала
    merged_ch1 = merge_intervals(table['канал 1'])
    merged_ch2 = merge_intervals(table['канал 2'])

    # 1. Время простоя хотя бы одного канала (когда один свободен, а второй может быть занят)
    # Это сумма времени простоя каждого канала минус время простоя обоих одновременно

    # Время работы каждого канала
    busy_ch1 = sum(end - start for start, end in merged_ch1)
    busy_ch2 = sum(end - start for start, end in merged_ch2)

    # Время простоя каждого канала
    idle_ch1 = time_N - busy_ch1
    idle_ch2 = time_N - busy_ch2

    # 2. Время простоя обоих каналов одновременно (система полностью свободна)
    # Находим интервалы, когда оба канала свободны
    free_ch1 = get_free_intervals(merged_ch1, time_N)
    free_ch2 = get_free_intervals(merged_ch2, time_N)
    idle_both_intervals = get_overlap(free_ch1, free_ch2)
    idle_both = sum(end - start for start, end in idle_both_intervals)

    # 3. Время простоя хотя бы одного канала
    # Это когда либо первый свободен, либо второй, либо оба
    # Можно вычислить как: idle_ch1 + idle_ch2 - idle_both
    idle_at_least_one = idle_ch1 + idle_ch2 - idle_both

    # Вероятности
    P1 = idle_at_least_one / time_N  # Хотя бы один канал свободен
    P2 = idle_both / time_N          # Оба канала свободны (простой системы)
    Pc = P2                          # Для системы с двумя каналами Pc = P2

    print(f"Вероятность простоя хотя бы одного канала: {round(P1*100, 3)} %")
    print(f"Вероятность простоя двух каналов одновременно: {round(P2*100, 3)} %")
    print(f"Вероятность простоя всей системы: {round(Pc*100, 3)} %")

    return P1, P2, Pc

def calculate_queue_probabilities(queue_intervals_list, time_N):
    """
    Вычисляет вероятности для различных состояний очереди (от 0 до N заявок).

    :param queue_intervals_list: список списков кортежей (start, end), где каждый список - это заявки в очереди.
    :param time_N: общее время наблюдения.
    :return: словарь {P0з, P1з, ..., Pnз} - вероятности различных состояний очереди.
    """
    from itertools import combinations, chain



    num_places = len(queue_intervals_list)  # Количество мест в очереди

    # Объединяем интервалы для каждой заявки в очереди
    merged_intervals = [merge_intervals(intervals) for intervals in queue_intervals_list]

    # Вычисляем вероятности для каждого количества занятых мест в очереди
    state_durations = {i: 0 for i in range(num_places + 1)}

    # Анализируем моменты времени, когда занято i мест в очереди
    for i in range(1, num_places + 1):
        total_overlap_time = 0
        for combo in combinations(merged_intervals, i):
            overlap_intervals = combo[0]
            for other_intervals in combo[1:]:
                overlap_intervals = get_overlap(overlap_intervals, other_intervals)

            total_overlap_time += sum(end - start for start, end in overlap_intervals)

        # Коррекция времени, если превышает time_N
        state_durations[i] = min(total_overlap_time, time_N)

    # Вычисляем вероятность, что очередь пуста (P0з)
    total_busy_time = sum(state_durations[i] for i in range(1, num_places + 1))
    state_durations[0] = max(0, time_N - total_busy_time)  # Простои системы

    # Коррекция суммарного времени, если оно превысило время наблюдения
    total_time_used = sum(state_durations.values())
    if total_time_used > time_N:
        scale_factor = time_N / total_time_used
        state_durations = {i: duration * scale_factor for i, duration in state_durations.items()}

    # Нормируем вероятности
    probabilities = [ state_durations[i] / time_N for i in range(num_places + 1)]

    # Выводим результат с правильным склонением
    def correct_word(n):
        if n == 1:
            return "заявка"
        elif 2 <= n <= 4:
            return "заявки"
        else:
            return "заявок"

    for i, p in enumerate(probabilities):
        print(f"Вероятность того, что в очереди будет {i} {correct_word(i)}: {round(p*100, 3)}%")

    return probabilities

def average_number_car_queue10(probabilities):
    i=0
    buf=0.0
    for p in probabilities:
        buf+=p*i
        i+=1

    print(f"Среднее количество заявок в очереди: {round(buf,3)} [шт]")
    return buf

def average_waiting_time13(queue_intervals_list, total_requests):
    from itertools import chain
    # Собираем все интервалы из всех мест в очереди
    all_intervals = list(chain.from_iterable(queue_intervals_list))
    merged_intervals = merge_intervals(all_intervals)
    total_waiting_time = sum(end - start for start, end in merged_intervals)
    # print(total_waiting_time)
    buf= total_waiting_time / total_requests if total_requests > 0 else 0
    print(f"Среднее время ожидания заявки в очереди: {round(buf,3)} часа ({round(buf,3)*60})")

    return buf
def average_service_time14(channel_intervals_list, total_requests):
    from itertools import chain

    # Собираем все интервалы из всех мест в очереди
    all_intervals = list(chain.from_iterable(channel_intervals_list))
    merged_intervals = merge_intervals(all_intervals)
    total_waiting_time = sum(end - start for start, end in merged_intervals)
    # print(total_waiting_time)
    buf= total_waiting_time / total_requests if total_requests > 0 else 0
    print(f"Среднее время обслуживания заявки: {round(buf,3)} часа ({round(buf,3)*60})")

    return buf
def average_application_time15(waiting, service):
    buf=waiting+service
    print(f"Среднее время нахождения заявки в системе: {round(buf,3)} часа ({round(buf,3)*60})")

    return buf
def average_applications_in_system_interval16(table,end_visor,start_visor,time_N,interval_length=1/6):
    """
    Вычисляет среднее количество заявок в системе методом разбиения на интервалы.

    :param table: словарь с данными о системе
    :param interval_length: длина подынтервала в часах (по умолчанию 10 минут = 1/6 часа)
    :return: среднее количество заявок в системе
    """
    # Получаем общее время наблюдения

    max_time = end_visor
    min_time = start_visor
    total_time = time_N

    # Количество подынтервалов
    K = int(total_time / interval_length) + 1

    # Создаем список моментов времени для проверки
    check_points = [min_time + i * interval_length for i in range(K)]
    check_points.append(max_time)  # Добавляем конечную точку

    # Функция для определения количества заявок в системе в момент времени t
    def apps_in_system_at_time(t):
        count = 0

        # Заявки в очередях
        for queue in ['очередь 1', 'очередь 2', 'очередь 3']:
            if queue in table:
                for start, end in table[queue]:
                    if start <= t < end:
                        count += 1

        # Заявки в каналах обслуживания
        for channel in ['канал 1', 'канал 2']:
            for start, end in table[channel]:
                if start <= t < end:
                    count += 1

        return count

    total_apps = 0
    prev_time = check_points[0]
    prev_count = apps_in_system_at_time(prev_time)

    for current_time in check_points[1:]:
        interval_length = current_time - prev_time
        total_apps += prev_count * interval_length
        prev_time = current_time
        prev_count = apps_in_system_at_time(current_time)

    average = total_apps / total_time if total_time > 0 else 0

    print(f"Среднее количество заявок в системе (метод интервалов): {round(average, 3)} [шт]\tИспользовано {K} интервалов по {round(interval_length*60,1)} минут")
    return average


Тестовый метод

In [8]:
def calculate_queue_probabilities(queue_intervals_list, time_N):
    """
    Вычисляет вероятности для различных состояний очереди (от 0 до N заявок).

    :param queue_intervals_list: список списков кортежей (start, end), где каждый список - это заявки в очереди
    :param time_N: общее время наблюдения
    :return: список вероятностей [P0, P1, ..., Pn] для состояний очереди
    """
    # Количество мест в очереди
    num_places = len(queue_intervals_list)

    # Объединяем и сортируем интервалы для каждого места в очереди
    merged_intervals = [merge_intervals(intervals) for intervals in queue_intervals_list]

    # Собираем все точки изменения состояния очереди
    events = []
    for place, intervals in enumerate(merged_intervals):
        for start, end in intervals:
            events.append((start, 'arrival', place))
            events.append((end, 'departure', place))

    # Сортируем события по времени
    events.sort(key=lambda x: x[0])

    # Инициализируем переменные для подсчета
    current_time = 0
    queue_state = 0  # Текущее количество заявок в очереди
    state_durations = [0.0] * (num_places + 1)

    # Обрабатываем события
    for time, event_type, _ in events:
        # Добавляем время текущего состояния
        if current_time < time:
            state_durations[queue_state] += time - current_time
            current_time = time

        # Обновляем состояние очереди
        if event_type == 'arrival':
            if queue_state < num_places:
                queue_state += 1
        elif event_type == 'departure':
            if queue_state > 0:
                queue_state -= 1

    # Добавляем последний интервал
    if current_time < time_N:
        state_durations[queue_state] += time_N - current_time

    # Вычисляем вероятности
    total_time = sum(state_durations)
    probabilities = [duration / total_time for duration in state_durations]

    # Выводим результаты с правильным склонением
    def plural_form(n):
        if n % 10 == 1 and n % 100 != 11:
            return "заявка"
        elif 2 <= n % 10 <= 4 and (n % 100 < 10 or n % 100 >= 20):
            return "заявки"
        else:
            return "заявок"

    for i, prob in enumerate(probabilities):
        print(f"Вероятность {i} {plural_form(i)} в очереди: {round(prob*100, 3)}%")

    return probabilities

In [9]:
def get_table_params(table1):
    params=[]
    start_visor=table1['обслуженно'][0][0][0]
    end_visor=table1['обслуженно'][-1][0][1]
    time_N=end_visor-start_visor
    print(f"Время наблюдения с {start_visor} по {end_visor} равно {time_N}")
    params.append(probability_service1(table1))
    params.append(system_throughput2(table1,time_N))
    params.append(probability_failure3(table1))
    sco=single_channel_occupancy45(table1,time_N)
    params.append(sco)
    params.append(average_occupied_channels6(sco))
    params.append(calculate_idle_probabilities789(table1,time_N))
    cqp=calculate_queue_probabilities([values for values in list(table1.values())[5:]],time_N)

    params.append(cqp)
    params.append(average_number_car_queue10(cqp))


    w=average_waiting_time13([values for values in list(table1.values())[5:]],len(table1['запросы']))
    s=average_service_time14([values for values in list(table1.values())[1:3]],len(table1['канал 1'])+len(table1['канал 2']))
    params.append(w)
    params.append(s)
    params.append(average_application_time15(w,s))
    params.append(average_applications_in_system_interval16(table1,end_visor,start_visor,time_N))

    b=[]
    for i in params:
        if isinstance(i, (list, tuple, dict)):
            for j in i:
                b.append(j)
        else:
            b.append(i)
    params=b
    return params




def run(n_queue,req=None):
    if req is None:
        requests=[round(generator_gap(lamd),3)]
        for i in range(0,N,1):
            requests.append(round(requests[-1]+generator_gap(lamd),3))
    else:
        requests=req

    table, request_objects = service_class(requests, n_queue ,kanal_mu)

    table1=filter_table_by_time_range(table,table["запросы"][-1]*0.05,table["запросы"][-1]*0.95)
    # print_table(table)
    # print()
    # print_table(table1)

    start_visor=table1['обслуженно'][0][0][0]
    end_visor=table1['обслуженно'][-1][0][1]
    time_N=end_visor-start_visor

    p=get_table_params(table1)
    # visualize_service_system(table)
    # visualize_service_system(table1)
    # print_table(table)
    # print()
    # print_table(table1)

    data={
        "Начальные данные": table,
        "Данные для исследования":table1,
        "Контрольные значения таблицы": p,
        "Время наблюдения": time_N,
        "Начало наблюдения":start_visor,
        "Конец наблюдения": end_visor
    }
    return data


начальные параметры

In [10]:
lamd=10
mu_1=2
mu_2=8
kanal_mu={
    "канал 1": mu_1,
    "канал 2": mu_2
}
queue_len=3
N=150


In [11]:
import json

def load_dict_from_file(file_path):
    """
    Читает словарь из файла JSON.

    Параметры:
        file_path (str): Путь к файлу (например, 'data.json')

    Возвращает:
        dict: Загруженный словарь
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        dictionary = json.load(file)
    print(f"Словарь загружен из файла: {file_path}")
    return dictionary

def save_dict_to_file(dictionary, file_path):
    """
    Сохраняет словарь в файл в формате JSON.

    Параметры:
        dictionary (dict): Словарь для сохранения
        file_path (str): Путь к файлу (например, 'data.json')
    """
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(dictionary, file, ensure_ascii=False, indent=4)
    print(f"Словарь сохранён в файл: {file_path}")

генерируем данные

In [12]:
# table_data={}
# requests=[round(generator_gap(lamd),3)]
# for i in range(0,N,1):
#     requests.append(round(requests[-1]+generator_gap(lamd),3))
# for i in range(queue_len+1):
#     print(f"\nКоличество мест в очереди {i}")
#     table_data[f"Количество мест в очереди {i}"]=run(i,requests)

table_data=load_dict_from_file("cmo.json")

Словарь загружен из файла: cmo.json


табличка с расчитанной статистикой

In [21]:
params_table=[]
params_table=[get_table_params(i["Данные для исследования"]) for i in table_data.values()]
metrics = [
    "Вероятность обслуживания",
    "Пропускная способность системы",
    "Вероятность отказа",
    "Вероятность занятости одного канала",
    "Вероятность занятости двух каналов",
    "Среднее количество занятых каналов",
    "Вероятность простоя хотя бы одного канала",
    "Вероятность простоя двух каналов одновременно",
    "Вероятность простоя всей системы",
    "Вероятность 0 заявок в очереди",

    "Среднее количество заявок в очереди",
    "Среднее время ожидания заявки в очереди",
    "Среднее время обслуживания заявки",
    "Среднее время нахождения заявки в системе",
    "Среднее количество заявок в системе (метод интервалов)"
]


# for i in params_table:
#     if len(i)<len(params_table[-1]):
#         for _ in range(queue_len+1-i[9]):
#             i.insert(10+i[9],0)
#     i.remove(i[9])
#     print(len(i),i)
# if len(metrics)<len(params_table[-1]):
#     for i in range(queue_len,0,-1):
#         metrics.insert(10,f"Вероятность {i} заявок в очереди")
# for i in range(len(metrics)):
#     print(metrics[i],end="\t")
#     for j in params_table:
#         print(round(j[i],3),end="\t")
#     print("\b")
# queue_sizes=[i for i in range(queue_len+1)]
# # Создаем словарь для удобного доступа к данным
# data = {metric: [] for metric in metrics}
# for i, metric in enumerate(metrics):
#     for j in range(len(queue_sizes)):
#         if i < len(params_table[j]):
#             data[metric].append(params_table[j][i])

Время наблюдения с 1.174 по 13.037 равно 11.863000000000001
Вероятность обслуживания: 49.3 %
Пропускная способность системы: 5.564 [шт/час]
Вероятность отказа: 50.0 %
Вероятность занятости одного канала: 35.84253561493713 %
Вероятность занятости двух канала: 55.68574559554921 %
Среднее количество занятых каналов: 1.4721402680603555 канала	Показатель загрузки: 0.7360701340301777%
Вероятность простоя хотя бы одного канала: 32.597 %
Вероятность простоя двух каналов одновременно: 20.189 %
Вероятность простоя всей системы: 20.189 %
Вероятность 0 заявок в очереди: 100.0%
Среднее количество заявок в очереди: 0.0 [шт]
Среднее время ожидания заявки в очереди: 0.0 часа (0.0)
Среднее время обслуживания заявки: 0.162 часа (9.72)
Среднее время нахождения заявки в системе: 0.162 часа (9.72)
Среднее количество заявок в системе (метод интервалов): 1.424 [шт]	Использовано 72 интервалов по 1.8 минут
Время наблюдения с 0.791 по 13.003 равно 12.212
Вероятность обслуживания: 63.4 %
Пропускная способность с

In [16]:
params_table

[[0.493,
  5.564,
  0.5,
  0.35842535614937127,
  0.5568574559554921,
  1.4721402680603555,
  0.32597150805024017,
  0.20188822388940422,
  0.20188822388940422,
  1.0,
  0.0,
  0.0,
  0.16205970149253726,
  0.16205970149253726,
  1.4239793194526393],
 [0.634,
  6.96,
  0.366,
  0.2810350474942683,
  0.632165083524402,
  1.5453652145430723,
  0.3030625614150019,
  0.15157222404192586,
  0.15157222404192586,
  0.6534861956013103,
  0.34651380439868973,
  0.34651380439868973,
  0.033156716417910445,
  0.1312,
  0.16435671641791044,
  1.9416966917785787],
 [0.657,
  7.332,
  0.336,
  0.1385602399600064,
  0.8266122312947846,
  1.7917847025495754,
  0.07523746042326243,
  0.13297783702716212,
  0.13297783702716212,
  0.3733731434696065,
  0.2753024039197669,
  0.3513244526106266,
  0.9779513091410201,
  0.06108208955223881,
  0.1287111111111111,
  0.18979320066334993,
  2.8889074043215026],
 [0.821,
  9.002,
  0.194,
  0.10941080196399344,
  0.8728314238952538,
  1.855073649754501,
  0.0612

In [ ]:
# Оптимизированная обработка данных
def prepare_metrics_data(params_table, metrics, queue_len):
    max_len = len(params_table[-1])
    
    # Дополняем метрики при необходимости
    if len(metrics) < max_len:
        missing=max_len - len(metrics)
        for i in range(missing, 0, -1):
            metrics.insert(10, f"Вероятность {i} заявок в очереди")
    # Нормализация таблицы параметров
    max_len = len(params_table[-1])
    i=0
    for row in params_table:
        # Заполняем недостающие значения нулями
        if len(row) < max_len:
            missing = max_len - len(row)
            for i in range(missing):
            row[10+i] = 0   # Вставляем нули после 10-го элемента
            i+=1
        # Удаляем 9-й элемент (индекс 8, так как индексация с 0)




    # Создаем словарь данных
    queue_sizes = list(range(queue_len + 1))
    data = {}

    for i, metric in enumerate(metrics):
        metric_values = []
        for j in range(len(queue_sizes)):
            if i < len(params_table[j]):
                metric_values.append(params_table[j][i])
        data[metric] = metric_values

    return data, queue_sizes



# Оптимизированная функция вывода таблицы
def print_metrics_table(metrics, params_table,queue_sizes):
    # Определяем ширину колонок
    metric_width = max(len(m) for m in metrics) + 2
    value_width = 10

    # Заголовок с размерами очереди
    header = "Метрика".ljust(metric_width)
    for size in queue_sizes:
        header += f"{size} мест".center(value_width)
    print(header)
    print("-" * len(header))

    # Вывод данных
    for i, metric in enumerate(metrics):
        row = metric.ljust(metric_width)
        for j in range(len(queue_sizes)):
            if i < len(params_table[j]):
                value = f"{params_table[j][i]:.3f}"
                row += value.center(value_width)
        print(row)
data, queue_sizes = prepare_metrics_data(params_table, metrics, queue_len)
# Вывод таблицы
for i,j in zip(data.keys(),data.values()):
    print(f"{i} \t {j}")
    

# print_metrics_table(metrics, params_table, queue_sizes)

Вероятность обслуживания 	 [0.493, 0.634, 0.657, 0.821]
Пропускная способность системы 	 [5.564, 6.96, 7.332, 9.002]
Вероятность отказа 	 [0.5, 0.366, 0.336, 0.194]
Вероятность занятости одного канала 	 [0.35842535614937127, 0.2810350474942683, 0.1385602399600064, 0.10941080196399344]
Вероятность занятости двух каналов 	 [0.5568574559554921, 0.632165083524402, 0.8266122312947846, 0.8728314238952538]
Среднее количество занятых каналов 	 [1.4721402680603555, 1.5453652145430723, 1.7917847025495754, 1.855073649754501]
Вероятность простоя хотя бы одного канала 	 [0.32597150805024017, 0.3030625614150019, 0.07523746042326243, 0.06121112929623554]
Вероятность простоя двух каналов одновременно 	 [0.20188822388940422, 0.15157222404192586, 0.13297783702716212, 0.08371522094926342]
Вероятность простоя всей системы 	 [0.20188822388940422, 0.15157222404192586, 0.13297783702716212, 0.08371522094926342]
Вероятность 0 заявок в очереди 	 [1.0, 0.6534861956013103, 0.3733731434696065, 0.3080800248138956]


In [ ]:
from matplotlib.ticker import PercentFormatter
from matplotlib import font_manager
def plot_enhanced_metrics(data,queue_sizes):
    """
    Визуализирует метрики системы массового обслуживания.

    :param metrics: список названий метрик
    :param params_table: таблица значений (список списков)
    :param queue_sizes: список размеров очередей
    """
    # queue_sizes=[i for i in range(iteration_N+1)]
    # # Создаем словарь для удобного доступа к данным
    # data = {metric: [] for metric in metrics}
    # for i, metric in enumerate(metrics):
    #     for j in range(len(queue_sizes)):
    #         if i < len(params_table[j]):
    #             data[metric].append(params_table[j][i])

    try:
        plt.style.use('seaborn-v0_8')  # Новое название стиля в современных версиях
    except:
        plt.style.use('ggplot')  # Альтернативный хороший стиль

    # Проверяем доступность шрифта
    try:
        # Получаем список доступных шрифтов
        available_fonts = set(f.name for f in font_manager.fontManager.ttflist)
        safe_font = 'DejaVu Sans' if 'DejaVu Sans' in available_fonts else 'Arial'
        plt.rcParams['font.family'] = safe_font
    except:
        plt.rcParams['font.family'] = 'sans-serif'  # Резервный вариант

    # Группировка метрик
    metric_groups = {
        "Эффективность системы": [
            ("Вероятность обслуживания", "Процент", True),
            ("Пропускная способность системы", "шт/час", False),
            ("Вероятность отказа", "Процент", True)
        ],
        "Загрузка каналов": [
            ("Вероятность занятости одного канала", "Процент", True),
            ("Вероятность занятости двух каналов", "Процент", True),
            ("Среднее количество занятых каналов", "каналы", False)
        ],
        "Состояния системы": [
            ("Вероятность простоя хотя бы одного канала", "Процент", True),
            ("Вероятность простоя двух каналов одновременно", "Процент", True),
            ("Вероятность простоя всей системы", "Процент", True)
        ],
        "Характеристики очереди": [
            ("Среднее количество заявок в очереди", "заявки", False),
            ("Среднее время ожидания заявки в очереди", "часы", False),
            ("Среднее время обслуживания заявки", "часы", False),
            ("Среднее время нахождения заявки в системе", "часы", False)
        ],
        "Вероятности состояний очереди": [
            (m, "Процент", True) for m in metrics if m.startswith("Вероятность") and "заявок" in m
        ]
    }

    # Построение графиков
    for group_name, group_metrics in metric_groups.items():
        if not group_metrics:
            continue

        fig, axes = plt.subplots(1, len(group_metrics), figsize=(15, 4))
        if len(group_metrics) == 1:
            axes = [axes]

        fig.suptitle(group_name, fontsize=14, y=1.05)

        for ax, (metric, unit, is_percent) in zip(axes, group_metrics):
            if metric not in data:
                continue

            values = np.array(data[metric])
            ax.plot(queue_sizes, values, 'o-', linewidth=2, markersize=8,
                   color='#2c7bb6', markerfacecolor='white')

            ax.set_title(metric, fontsize=12)
            ax.set_xlabel('Количество мест в очереди', fontsize=10)
            ax.set_ylabel(unit, fontsize=10)
            ax.grid(True, linestyle=':', alpha=0.7)

            if is_percent:
                ax.yaxis.set_major_formatter(PercentFormatter(1.0))

            # Добавление значений на график
            for x, y in zip(queue_sizes, values):
                label = f"{y:.1%}" if is_percent else f"{y:.3f}"
                ax.text(x, y, label, ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

    # Отдельный график для среднего количества заявок в системе
    if "Среднее количество заявок в системе (метод интервалов)" in data:
        fig, ax = plt.subplots(figsize=(8, 5))
        metric = "Среднее количество заявок в системе (метод интервалов)"
        values = data[metric]

        ax.plot(queue_sizes, values, 's--', linewidth=2, markersize=10,
               color='#d7191c', markerfacecolor='white')

        ax.set_title(metric, fontsize=12)
        ax.set_xlabel('Количество мест в очереди', fontsize=10)
        ax.set_ylabel('Заявки', fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.7)

        for x, y in zip(queue_sizes, values):
            ax.text(x, y, f"{y:.3f}", ha='center', va='bottom', fontsize=10)

        plt.tight_layout()
        plt.show()
# Пример использования:
data, queue_sizes = prepare_metrics_data(params_table, metrics, queue_len)
# Пример использования:
plot_enhanced_metrics(data, queue_sizes)

In [ ]:
def plot_enhanced_metrics(data, queue_sizes):
    """
    Визуализирует метрики системы массового обслуживания.

    :param data: словарь с метриками (ключи - названия метрик)
    :param queue_sizes: список размеров очередей
    """
    try:
        plt.style.use('seaborn-v0_8')
    except:
        plt.style.use('ggplot')

    try:
        available_fonts = set(f.name for f in font_manager.fontManager.ttflist)
        safe_font = 'DejaVu Sans' if 'DejaVu Sans' in available_fonts else 'Arial'
        plt.rcParams['font.family'] = safe_font
    except:
        plt.rcParams['font.family'] = 'sans-serif'

    metric_groups = {
        "Эффективность системы": [
            ("Вероятность обслуживания", "Процент", True),
            ("Пропускная способность системы", "шт/час", False),
            ("Вероятность отказа", "Процент", True)
        ],
        "Загрузка каналов": [
            ("Вероятность занятости одного канала", "Процент", True),
            ("Вероятность занятости двух каналов", "Процент", True),
            ("Среднее количество занятых каналов", "каналы", False)
        ],
        "Состояния системы": [
            ("Вероятность простоя хотя бы одного канала", "Процент", True),
            ("Вероятность простоя двух каналов одновременно", "Процент", True),
            ("Вероятность простоя всей системы", "Процент", True)
        ],
        "Характеристики очереди": [
            ("Среднее количество заявок в очереди", "заявки", False),
            ("Среднее время ожидания заявки в очереди", "часы", False),
            ("Среднее время обслуживания заявки", "часы", False),
            ("Среднее время нахождения заявки в системе", "часы", False)
        ],
        "Вероятности состояний очереди": [
            (m, "Процент", True) for m in data.keys() if m.startswith("Вероятность") and "заявок" in m
        ]
    }

    for group_name, group_metrics in metric_groups.items():
        if not group_metrics:
            continue

        fig, axes = plt.subplots(1, len(group_metrics), figsize=(15, 4))
        if len(group_metrics) == 1:
            axes = [axes]

        fig.suptitle(group_name, fontsize=14, y=1.05)

        for ax, (metric, unit, is_percent) in zip(axes, group_metrics):
            if metric not in data:
                continue

            values = np.array(data[metric])

            # Добавляем точку (0,0) если ее нет в данных
            if 0 not in queue_sizes:
                queue_sizes = [0] + queue_sizes
                values = np.concatenate(([0], values))

            ax.plot(queue_sizes, values, 'o-', linewidth=2, markersize=8,
                   color='#2c7bb6', markerfacecolor='white')

            ax.set_title(metric, fontsize=12)
            ax.set_xlabel('Количество мест в очереди', fontsize=10)
            ax.set_ylabel(unit, fontsize=10)

            # Устанавливаем границы осей, чтобы включить (0,0)
            ax.set_xlim(left=0)
            ax.set_ylim(bottom=0)

            ax.grid(True, linestyle=':', alpha=0.7)

            if is_percent:
                ax.yaxis.set_major_formatter(PercentFormatter(1.0))

            for x, y in zip(queue_sizes, values):
                label = f"{y:.1%}" if is_percent else f"{y:.3f}"
                ax.text(x, y, label, ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plt.show()

    # Отдельный график для среднего количества заявок в системе
    if "Среднее количество заявок в системе (метод интервалов)" in data:
        fig, ax = plt.subplots(figsize=(8, 5))
        metric = "Среднее количество заявок в системе (метод интервалов)"
        values = data[metric]

        # Добавляем точку (0,0) если ее нет в данных
        if 0 not in queue_sizes:
            queue_sizes = [0] + queue_sizes
            values = np.concatenate(([0], values))

        ax.plot(queue_sizes, values, 's--', linewidth=2, markersize=10,
               color='#d7191c', markerfacecolor='white')

        ax.set_title(metric, fontsize=12)
        ax.set_xlabel('Количество мест в очереди', fontsize=10)
        ax.set_ylabel('Заявки', fontsize=10)

        # Устанавливаем границы осей, чтобы включить (0,0)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)

        ax.grid(True, linestyle=':', alpha=0.7)

        for x, y in zip(queue_sizes, values):
            ax.text(x, y, f"{y:.3f}", ha='center', va='bottom', fontsize=10)

        plt.tight_layout()
        plt.show()
data, queue_sizes = prepare_metrics_data(params_table, metrics, queue_len)
# Пример использования:
plot_enhanced_metrics(data, queue_sizes)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_extremum(list1, list2, label1='Список 1', label2='Список 2', title='Экстремум', operation='sum'):
    """
    Визуализирует экстремум при сложении/вычитании двух списков.

    Параметры:
        list1: Первый список значений
        list2: Второй список значений
        label1: Подпись для первого списка
        label2: Подпись для второго списка
        title: Заголовок графика
        operation: 'sum' или 'diff' (сложение или вычитание)
    """
    if len(list1) != len(list2):
        raise ValueError("Списки должны быть одинаковой длины")

    # Вычисляем комбинированную метрику
    if operation == 'sum':
        combined = np.array(list1) + np.array(list2)
        combined/=2
        op_label = 'Сумма'
    elif operation == 'diff':
        combined = np.array(list1) - np.array(list2)
        combined/=2
        op_label = 'Разность'
    else:
        raise ValueError("Операция должна быть 'sum' или 'diff'")

    # Находим экстремум
    extreme_idx = np.argmax(combined)
    extreme_val = combined[extreme_idx]
    x_values = np.arange(len(list1))

    # Создаем график
    plt.figure(figsize=(10, 6))

    # Графики исходных данных
    plt.plot(x_values, list1, 'bo-', label=label1)
    plt.plot(x_values, list2, 'ro-', label=label2)

    # График комбинированной метрики
    plt.plot(x_values, combined, 'g.--', label=f'{op_label} ({label1} и {label2})')

    # Отмечаем экстремум
    plt.scatter(extreme_idx, combined[extreme_idx], s=200, color='gold', alpha=0.5,
               label=f'Экстремум ({extreme_idx}: {extreme_val:.2f})')
    plt.axvline(x=extreme_idx, color='gray', linestyle=':', alpha=0.5)

    # Настройки отображения
    plt.title(title)
    plt.xlabel('Индекс')
    plt.ylabel('Значение')
    plt.xticks(x_values)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.legend()

    plt.tight_layout()
    plt.show()

    return extreme_idx, extreme_val
data

In [ ]:

# Пример использования:
prob_1 = data['Вероятность простоя всей системы']  # Вероятность простоя
prob_2 = data['Вероятность занятости двух каналов'] # Вероятность занятости

idx, val = plot_extremum(
    prob_1,
    prob_2,
    label1='Вероятность простоя',
    label2='Вероятность занятости',
    title='Баланс между простом и занятостью',
    operation='sum'
)

print(f"Экстремум найден при индексе {idx} со значением {val:.3f}")

In [ ]:
prob_1 = data['Вероятность занятости двух каналов']  # Вероятность простоя
prob_2 = data['Вероятность отказа'] # Вероятность занятости

idx, val = plot_extremum(
    prob_1,
    prob_2,
    label1='Вероятность занятости двух каналов',
    label2='Вероятность отказа',
    title='',
    operation='diff'
)

print(f"Экстремум найден при индексе {idx} со значением {val:.3f}")

In [ ]:
prob_1 = data['Пропускная способность системы']  # Вероятность простоя
prob_2 = data['Среднее время обслуживания заявки'] # Вероятность занятости
prob_2=[i*60 for i in prob_2]

idx, val = plot_extremum(
    prob_1,
    prob_2,
    label1='пропускная способность',
    label2='Время обслуживания',
    title='',
    operation='diff'
)

print(f"Экстремум найден при индексе {idx} со значением {val:.3f}")

In [ ]:
# Пример использования:
prob_1 = data['Вероятность простоя всей системы']  # Вероятность простоя
prob_2 = data['Вероятность занятости двух каналов'] # Вероятность занятости

idx, val = plot_extremum(
    prob_1,
    prob_2,
    label1='Вероятность простоя',
    label2='Вероятность занятости',
    title='Баланс между простом и занятостью',
    operation='sum'
)

print(f"Экстремум найден при индексе {idx} со значением {val:.3f}")

In [ ]:
save_dict_to_file(table_data,"cmo.json")

In [ ]:
for data in table_data.values():
    print(data)

In [ ]:
for data,key in zip(table_data.values(),table_data.keys()):
    table=data['Начальные данные']
    table1=data['Данные для исследования']
    print(key,'\nНачальные данные')
    print_table(table)
    visualize_service_system(table)
    print('\nВыборка данных')

    print_table(table1)
    visualize_service_system(table1)

    print("\n")